# Decisions Playbook — Supporting Calculations
**Group 10:** Mohammed Saalif Udyawar · Jakkana Hasini · Mohammed Mutee Ruknuddin

This notebook reproduces every quarterly, product, store, inventory and savings figure quoted in `Group10_MUC01_KLU_Decisions.docx`. Section numbers (**C1–C8**) match the "Source" column in that document.

**How to run:** keep `MUC01_Retail_Sales_Dataset.csv` in this folder or its parent folder, install `requirements.txt`, then *Kernel → Restart Kernel and Run All Cells*.

**Definitions used throughout**
* **Q1** = January–March 2026, **Q2** = April–June 2026. *Q2 vs Q1* = (Q2 total ÷ Q1 total − 1) × 100.
* **Jan → Jun** compares the single months of January and June.
* **Units** = sum of `units_sold`. **Discount given** = units × unit price − revenue.

In [1]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

# The dataset may sit next to this notebook or one folder up (repository root)
CANDIDATES = ["MUC01_Retail_Sales_Dataset.csv", "../MUC01_Retail_Sales_Dataset.csv"]
DATA_FILE = next((path for path in CANDIDATES if os.path.exists(path)), None)
if DATA_FILE is None:
    raise FileNotFoundError("Place 'MUC01_Retail_Sales_Dataset.csv' in this folder or its parent folder, then run all cells again.")

df = pd.read_csv(DATA_FILE, parse_dates=["date"])
df["month"] = df["date"].dt.month
df["quarter"] = np.where(df["month"] <= 3, "Q1", "Q2")
df["pre_discount"] = df["units_sold"] * df["unit_price"]
df["discount_given"] = df["pre_discount"] - df["revenue"]
df["weekend"] = df["date"].dt.dayofweek >= 5
pct = lambda new, old: (new / old - 1) * 100
print(f"{len(df):,} transactions, {df['date'].min():%d %b %Y} – {df['date'].max():%d %b %Y}")

107,836 transactions, 01 Jan 2026 – 30 Jun 2026


## C1 — Category revenue: 6-month total, Jan → Jun and Q2 vs Q1

In [2]:
g = df.groupby("category")
c1 = pd.DataFrame({
    "6-month revenue (₹ Cr)": g["revenue"].sum() / 1e7,
    "Share of revenue (%)": g["revenue"].sum() / df["revenue"].sum() * 100,
    "Jan → Jun (%)": pct(df[df.month == 6].groupby("category")["revenue"].sum(), df[df.month == 1].groupby("category")["revenue"].sum()),
    "Q2 vs Q1 (%)": pct(df[df.quarter == "Q2"].groupby("category")["revenue"].sum(), df[df.quarter == "Q1"].groupby("category")["revenue"].sum()),
    "Discount given (₹ L)": g["discount_given"].sum() / 1e5,
})
c1.round(2)

,6-month revenue (₹ Cr),Share of revenue (%),Jan → Jun (%),Q2 vs Q1 (%),Discount given (₹ L)
category,,,,,
Apparel,6.95,15.11,34.10,23.40,53.69
Electronics,22.65,49.27,-35.02,-19.53,175.66
Grocery,4.23,9.20,-3.55,1.68,33.13
Home & Kitchen,8.18,17.80,-18.49,-7.42,62.86
Personal Care,3.96,8.62,2.18,6.56,30.71


## C2 — Product revenue and units (all 25 products)

In [3]:
keys = ["category", "product_name"]
by = lambda mask, col: df[mask].groupby(keys)[col].sum()
c2 = pd.DataFrame({
    "Rev Q2 vs Q1 (%)": pct(by(df.quarter == "Q2", "revenue"), by(df.quarter == "Q1", "revenue")),
    "Revenue Jan (₹ L)": by(df.month == 1, "revenue") / 1e5,
    "Revenue Jun (₹ L)": by(df.month == 6, "revenue") / 1e5,
    "Units Jan": by(df.month == 1, "units_sold"),
    "Units Jun": by(df.month == 6, "units_sold"),
    "Units May": by(df.month == 5, "units_sold"),
    "Avg units / month (6 mo)": df.groupby(keys)["units_sold"].sum() / 6,
    "Discount given (₹ L)": df.groupby(keys)["discount_given"].sum() / 1e5,
    "6-month revenue (₹ L)": df.groupby(keys)["revenue"].sum() / 1e5,
})
c2["Units Jan → Jun (%)"] = pct(c2["Units Jun"], c2["Units Jan"])
c2["Units Jun vs May (%)"] = pct(c2["Units Jun"], c2["Units May"])
pd.set_option("display.max_rows", 50)
c2.round(1)

Rev Q2 vs Q1 (%)  Revenue Jan (₹ L)  \
category       product_name                                           
Apparel        Formal Shirt                 18.2               20.2   
               Jeans                        32.3               18.9   
               Kurta                        17.5               20.7   
               Saree                        19.7               21.1   
               T-Shirt                      29.9               19.7   
Electronics    Headphones                   -5.7               91.0   
               Laptop                      -14.3               94.9   
               Mobile Phone                -26.8               98.0   
               Smart TV                    -19.3               75.3   
               Tablet                      -30.3               97.6   
Grocery        Atta (5kg)                    0.1               15.0   
               Dal (1kg)                     2.0               15.0   
               Oil (1L)                      2.6               14.9   
               Rice (5kg)                    2.6               14.5   
               Sugar (1kg)                   1.2               13.7   
Home & Kitchen Dinner Set                  -11.1               30.3   
               Mixer Grinder               -14.0               30.3   
               Pressure Cooker              -1.1               29.1   
               Storage Box                  -2.0               31.1   
               Water Bottle                 -8.0               30.6   
Personal Care  Body Lotion                   6.9               13.0   
               Face Wash                     2.8               13.4   
               Shampoo                       6.0               14.0   
               Sunscreen                     7.5               12.9   
               Toothpaste                   10.0               12.6   

                                Revenue Jun (₹ L)  Units Jan  Units Jun  \
category       product_name                                               
Apparel        Formal Shirt                  27.4       1169       1589   
               Jeans                         26.5       1098       1529   
               Kurta                         24.4       1126       1414   
               Saree                         27.9       1149       1548   
               T-Shirt                       28.5       1136       1521   
Electronics    Headphones                    72.5        403        305   
               Laptop                        65.8        423        273   
               Mobile Phone                  42.1        421        213   
               Smart TV                      58.8        337        277   
               Tablet                        57.6        448        263   
Grocery        Atta (5kg)                    14.1       5007       4656   
               Dal (1kg)                     13.9       4945       4610   
               Oil (1L)                      14.2       4900       4666   
               Rice (5kg)                    14.4       4834       4845   
               Sugar (1kg)                   13.9       4555       4573   
Home & Kitchen Dinner Set                    22.0        776        570   
               Mixer Grinder                 23.7        823        681   
               Pressure Cooker               25.7        730        654   
               Storage Box                   27.6        843        711   
               Water Bottle                  24.3        854        633   
Personal Care  Body Lotion                   13.6       2199       2288   
               Face Wash                     13.2       2255       2204   
               Shampoo                       13.9       2352       2345   
               Sunscreen                     13.6       2143       2298   
               Toothpaste                    13.0       2198       2228   

                                Units May  Avg units / month (6 mo)  \
category       pr

## C3 — Apparel units by city, Q1 vs Q2

In [4]:
app = df[df.category == "Apparel"].pivot_table(index="store_city", columns="quarter", values="units_sold", aggfunc="sum")
app["Growth (%)"] = pct(app["Q2"], app["Q1"])
app.sort_values("Growth (%)", ascending=False).round(1)

quarter,Q1,Q2,Growth (%)
store_city,,,
Guntur,3452,4471,29.5
Kakinada,1997,2542,27.3
Tirupati,3251,4019,23.6
Nellore,2268,2744,21.0
Rajahmundry,2720,3277,20.5
Vijayawada,4001,4714,17.8


## C4 — Store revenue, Q2 vs Q1 (all categories and Electronics only)

In [5]:
store = df.pivot_table(index=["store_id", "store_city"], columns="quarter", values="revenue", aggfunc="sum")
elec = df[df.category == "Electronics"].pivot_table(index=["store_id", "store_city"], columns="quarter", values="revenue", aggfunc="sum")
c4 = pd.DataFrame({
    "All categories Q2 vs Q1 (%)": pct(store["Q2"], store["Q1"]),
    "Electronics Q2 vs Q1 (%)": pct(elec["Q2"], elec["Q1"]),
})
c4.sort_values("All categories Q2 vs Q1 (%)").round(1)

,,All categories Q2 vs Q1 (%),Electronics Q2 vs Q1 (%)
store_id,store_city,,
S07,Tirupati,-17.9,-33.6
S02,Vijayawada,-11.8,-25.9
S11,Kakinada,-10.8,-25.3
S03,Guntur,-10.5,-25.3
S06,Rajahmundry,-10.0,-23.9
S08,Tirupati,-9.1,-26.8
S04,Guntur,-9.0,-17.8
S10,Nellore,-7.9,-13.2
S01,Vijayawada,-5.8,-15.2


## C5 — Discounts: bills above 10%, and the effect of a 10% cap *(illustration)*

In [6]:
above = df[df.discount_pct > 10]
retained = above["pre_discount"] * (above["discount_pct"] - 10) / 100
print(f"Transactions with a discount above 10%: {len(above):,}")
print(f"Discount that would not have been given if capped at 10%: ₹{retained.sum() / 1e5:,.1f} L "
      f"(Electronics ₹{retained[above.category == 'Electronics'].sum() / 1e5:,.1f} L)")
print("Assumes the same transactions, units and prices would still occur — an illustration, not a forecast.")
share = df.groupby("discount_pct").size() / len(df) * 100
print("\nShare of transactions by discount level (%):", share.round(1).to_dict())

Transactions with a discount above 10%: 24,071
Discount that would not have been given if capped at 10%: ₹83.7 L (Electronics ₹42.0 L)
Assumes the same transactions, units and prices would still occur — an illustration, not a forecast.

Share of transactions by discount level (%): {0: 33.4, 5: 22.2, 10: 22.1, 15: 11.1, 20: 11.3}


## C6 — Weekend discounts and weekend share of units

In [7]:
print(f"Discount given on Saturdays and Sundays: ₹{df.loc[df.weekend, 'discount_given'].sum() / 1e7:,.2f} Cr")
wk = df.pivot_table(index="category", columns="weekend", values="units_sold", aggfunc="sum")
(wk[True] / wk.sum(axis=1) * 100).round(1).rename("Weekend share of units (%)").to_frame()

Discount given on Saturdays and Sundays: ₹1.32 Cr


,Weekend share of units (%)
category,
Apparel,36.0
Electronics,38.2
Grocery,36.0
Home & Kitchen,37.4
Personal Care,36.4


## C7 — Run-rate illustration: June level × 6 months vs first-half actual

In [8]:
first_half = df.groupby("category")["revenue"].sum()
june_x6 = df[df.month == 6].groupby("category")["revenue"].sum() * 6
c7 = pd.DataFrame({"First half actual (₹ Cr)": first_half / 1e7, "June × 6 (₹ Cr)": june_x6 / 1e7})
c7["Difference (₹ Cr)"] = c7["June × 6 (₹ Cr)"] - c7["First half actual (₹ Cr)"]
print("Simple arithmetic illustration — assumes June's revenue repeats for six months. Not a forecast.")
c7.round(2)

Simple arithmetic illustration — assumes June's revenue repeats for six months. Not a forecast.


,First half actual (₹ Cr),June × 6 (₹ Cr),Difference (₹ Cr)
category,,,
Apparel,6.95,8.09,1.14
Electronics,22.65,17.81,-4.84
Grocery,4.23,4.23,-0.00
Home & Kitchen,8.18,7.40,-0.78
Personal Care,3.96,4.04,0.08


## C8 — Transactions and average unit price, January vs June

In [9]:
jan, jun = df[df.month == 1], df[df.month == 6]
c8 = pd.DataFrame({
    "Transactions Jan": jan.groupby("category").size(),
    "Transactions Jun": jun.groupby("category").size(),
    "Avg unit price Jan (₹)": jan.groupby("category")["unit_price"].mean(),
    "Avg unit price Jun (₹)": jun.groupby("category")["unit_price"].mean(),
})
c8["Transactions change (%)"] = pct(c8["Transactions Jun"], c8["Transactions Jan"])
c8["Price change (%)"] = pct(c8["Avg unit price Jun (₹)"], c8["Avg unit price Jan (₹)"])
c8.round(1)

,Transactions Jan,Transactions Jun,Avg unit price Jan (₹),Avg unit price Jun (₹),Transactions change (%),Price change (%)
category,,,,,,
Apparel,2233,2931,1916.0,1914.0,31.3,-0.1
Electronics,790,526,24183.0,23658.6,-33.4,-2.2
Grocery,9328,9106,324.3,325.1,-2.4,0.2
Home & Kitchen,1575,1266,4020.1,4129.8,-19.6,2.7
Personal Care,4325,4413,638.2,637.8,2.0,-0.1
